**Bagian A: Menyiapkan Streaming Source**

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming source berhasil disiapkan.")
print("Status streaming:", df_stream.isStreaming)

Streaming source berhasil disiapkan.
Status streaming: True


**Bagian B: Agregasi per Cabang**

In [15]:
df_pesanan = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

agregasi_cabang = df_pesanan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Agregasi per cabang berhasil dijalankan.")

Agregasi per cabang berhasil dijalankan.


**Bagian C: Agregasi Berbasis Window Waktu**

In [16]:
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

df_stream_waktu = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

agregasi_waktu = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Agregasi window waktu berhasil dijalankan.")

Agregasi window waktu berhasil dijalankan.


**Bagian D: Menguji dengan Data Simulasi**

Batch pertama

In [17]:
from datetime import datetime

waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Matcha"],
    "qty": [2, 1, 3],
    "harga_satuan": [18000, 15000, 22000],
    "waktu_pesan": [waktu_sekarang] * 3
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch 1 dikirim.")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

Batch 1 dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
| Yogyakarta|           15000|             1|
|   Magelang|          102000|             2|
+-----------+----------------+--------------+



Batch kedua

In [5]:
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Solo", "Magelang", "Yogyakarta"],
    "nama_menu": ["Cappuccino", "Latte", "Kopi Susu"],
    "qty": [2, 1, 2],
    "harga_satuan": [25000, 20000, 18000],
    "waktu_pesan": [waktu_sekarang] * 3
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch 2 dikirim.")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

Batch 2 dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           50000|             1|
| Yogyakarta|           51000|             2|
|   Magelang|          122000|             3|
+-----------+----------------+--------------+



Batch ketiga

In [6]:
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

pd.DataFrame({
    "pesanan_id": ["P007", "P008", "P009"],
    "nama_cabang": ["Yogyakarta", "Solo", "Magelang"],
    "nama_menu": ["Matcha", "Americano", "Cappuccino"],
    "qty": [2, 3, 1],
    "harga_satuan": [22000, 15000, 25000],
    "waktu_pesan": [waktu_sekarang] * 3
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch 3 dikirim.")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

Batch 3 dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           95000|             2|
| Yogyakarta|           95000|             3|
|   Magelang|          147000|             4|
+-----------+----------------+--------------+



**Bagian E: Sink Berbasis Berkas (Parquet)**

In [8]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming file sink Parquet berhasil dijalankan.")

Streaming file sink Parquet berhasil dijalankan.


Mengirim batch tambahan

In [9]:
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu": ["Kopi Susu", "Matcha"],
    "qty": [2, 1],
    "harga_satuan": [18000, 22000],
    "waktu_pesan": [waktu_sekarang] * 2
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

print("Batch tambahan dikirim.")
time.sleep(8)

Batch tambahan dikirim.


Memeriksa hasil penyimpanan Parquet

In [10]:
!ls tugas10_output

part-00000-5439232c-d75f-45b4-979d-6304a7e59a19-c000.snappy.parquet
part-00000-8ff28ff2-5b95-4bb0-8101-61c81c6aeb10-c000.snappy.parquet
part-00001-d7741c0a-753d-4b72-919a-c87ac0b07f25-c000.snappy.parquet
part-00002-2ebb818d-e9ec-4daa-9fb2-6bf6908559f5-c000.snappy.parquet
_spark_metadata


Membaca kembali data Parquet

In [11]:
df_parquet = spark.read.parquet("tugas10_output")

df_parquet.show()

+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+----------+---+------------+-------------------+
|      P004|       Solo|Cappuccino|  2|       25000|2026-11-08 15:57:52|
|      P005|   Magelang|     Latte|  1|       20000|2026-11-08 15:57:52|
|      P006| Yogyakarta| Kopi Susu|  2|       18000|2026-11-08 15:57:52|
|      P001|   Magelang| Kopi Susu|  2|       18000|2026-11-08 15:57:15|
|      P002| Yogyakarta| Americano|  1|       15000|2026-11-08 15:57:15|
|      P003|   Magelang|    Matcha|  3|       22000|2026-11-08 15:57:15|
|      P007| Yogyakarta|    Matcha|  2|       22000|2026-11-08 15:58:41|
|      P008|       Solo| Americano|  3|       15000|2026-11-08 15:58:41|
|      P009|   Magelang|Cappuccino|  1|       25000|2026-11-08 15:58:41|
|      P010|   Magelang| Kopi Susu|  2|       18000|2026-11-08 16:00:06|
|      P011|       Solo|    Matcha|  1|       22000

Menghentikan seluruh streaming query

In [12]:
query1.stop()
query2.stop()
query3.stop()

spark.stop()

print("Seluruh streaming query berhasil dihentikan.")

Seluruh streaming query berhasil dihentikan.
